In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import datetime as dt
import seaborn as sns
import numpy as np
from mlxtend.frequent_patterns import association_rules, apriori
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import math
import warnings
warnings.filterwarnings("ignore")

In [4]:
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic') 
else:   
    plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

In [5]:
articles = pd.read_csv("articles.csv", encoding= 'utf-8')
customers = pd.read_csv("customers.csv", encoding= 'utf-8')
submission = pd.read_csv("sample_submission.csv", encoding= 'utf-8')
transactions = pd.read_csv("transactions_train.csv", encoding= 'utf-8')

In [6]:
transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [7]:
submission.head()

,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0706016001 0706016002 0372860001 0610776002 07...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0706016001 0706016002 0372860001 0610776002 07...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0706016001 0706016002 0372860001 0610776002 07...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0706016001 0706016002 0372860001 0610776002 07...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0706016001 0706016002 0372860001 0610776002 07...


In [8]:
customers.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [9]:
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


## 데이터 결합

In [10]:
# 1. 거래 데이터 + 상품 데이터 조인
df = pd.merge(transactions, articles, on='article_id', how='left')

# 2. 거래/상품 데이터 + 고객 데이터 조인
df = pd.merge(df, customers, on='customer_id', how='left')

# 3. 최종 결과에 Submission 테이블 조인
df = pd.merge(df, submission, on='customer_id', how='left')

print(df.head())

        t_dat                                        customer_id  article_id  \
0  2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   663713001   
1  2018-09-20  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   541518023   
2  2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...   505221004   
3  2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...   685687003   
4  2018-09-20  00007d2de826758b65a93dd24ce629ed66842531df6699...   685687004   

      price  sales_channel_id  product_code                 prod_name  \
0  0.050831                 2        663713  Atlanta Push Body Harlow   
1  0.030492                 2        541518   Rae Push (Melbourne) 2p   
2  0.015237                 2        505221               Inca Jumper   
3  0.016932                 2        685687      W YODA KNIT OL OFFER   
4  0.016932                 2        685687      W YODA KNIT OL OFFER   

   product_type_no product_type_name  product_group_name  ...  \
0              

## 값 수정

In [11]:
df['price'] = df['price'] * 590

df.loc[(df['FN'].isna()) & (df['fashion_news_frequency'] == 'Regularly'), 'FN'] = 1
df.loc[(df['FN'].isna()) & (df['fashion_news_frequency'] == 'Monthly'), 'FN'] = 1

## 칼럼 삭제

In [12]:
cols_to_drop = [
    'detail_desc', 'postal_code', 'product_code',
    'product_type_no', 'graphical_appearance_no', 'colour_group_code', 
    'perceived_colour_value_id', 'perceived_colour_master_id', 
    'department_no', 'index_code', 'index_group_no', 'section_no', 'garment_group_no'
]

df.drop(columns=cols_to_drop, inplace=True)

print(f"남은 칼럼 수: {len(df.columns)}")

남은 칼럼 수: 23


## 결측치 처리

In [13]:
df['FN'] = df['FN'].fillna(0)

df['Active'] = df['Active'].fillna(0)

df['club_member_status'] = df['club_member_status'].fillna('Unknown')

df['fashion_news_frequency'] = df['fashion_news_frequency'].replace('None', 'NONE')
df['fashion_news_frequency'] = df['fashion_news_frequency'].fillna('NONE')

df = df.dropna(subset=['age'])

## 타입 변환

In [14]:
# 1. Age (나이): 실수에서 정수로 변환
df['age'] = df['age'].astype(int)

# 2. 범주형 데이터
# 'club_member_status'나 'fashion_news_frequency'처럼 반복되는 글자는 'category' 타입으로 바꾸면 메모리 사용량이 80% 이상 줄어든다고 함
df['club_member_status'] = df['club_member_status'].astype('category')
df['fashion_news_frequency'] = df['fashion_news_frequency'].astype('category')

# 3. ID 칼럼: 문자열로 통일 ('0'이 잘리는 문제를 방지하기 위해 문자열로 고정)
df['article_id'] = df['article_id'].astype(str)
df['customer_id'] = df['customer_id'].astype(str)

# 4. 날짜 데이터: 문자열에서 날짜형으로 변환
df['t_dat'] = pd.to_datetime(df['t_dat'])


df['FN'] = df['FN'].astype(int)
df['Active'] = df['Active'].astype(int)

## 칼럼 생성

In [15]:
# 1. 구간 설정 (0~19세, 20~29세, ..., 59세~150세)
bins = [0, 20, 30, 40, 50, 60, 150]

# 2. 라벨 설정 (각 구간의 이름)
labels = ['10대', '20대', '30대', '40대', '50대', '60대 이상']

# 3. 새로운 칼럼 생성
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

# 확인
print(df[['age', 'age_group']].head())

   age age_group
0   24       20대
1   24       20대
2   32       30대
3   32       30대
4   32       30대


## 확인

In [16]:
df.info()

<class 'pandas.DataFrame'>
Index: 31648066 entries, 0 to 31788323
Data columns (total 24 columns):
 #   Column                        Dtype         
---  ------                        -----         
 0   t_dat                         datetime64[us]
 1   customer_id                   str           
 2   article_id                    str           
 3   price                         float64       
 4   sales_channel_id              int64         
 5   prod_name                     str           
 6   product_type_name             str           
 7   product_group_name            str           
 8   graphical_appearance_name     str           
 9   colour_group_name             str           
 10  perceived_colour_value_name   str           
 11  perceived_colour_master_name  str           
 12  department_name               str           
 13  index_name                    str           
 14  index_group_name              str           
 15  section_name                  str           
 

In [17]:
missing_counts = df.isnull().sum()

print(missing_counts[missing_counts == 0])

t_dat                           0
customer_id                     0
article_id                      0
price                           0
sales_channel_id                0
prod_name                       0
product_type_name               0
product_group_name              0
graphical_appearance_name       0
colour_group_name               0
perceived_colour_value_name     0
perceived_colour_master_name    0
department_name                 0
index_name                      0
index_group_name                0
section_name                    0
garment_group_name              0
FN                              0
Active                          0
club_member_status              0
fashion_news_frequency          0
age                             0
prediction                      0
age_group                       0
dtype: int64


## 필요 칼럼만 뽑아서 저장

In [18]:
# 1. 분석용 핵심 칼럼 리스트 정의
target_cols = [
    't_dat', 'customer_id', 'price', 'age', 'Active', 'FN', 
    'club_member_status', 'age_group', 'article_id', 'prod_name', 
    'product_group_name', 'section_name', 'garment_group_name'
]

# 2. 존재하는 칼럼만 필터링하여 새로운 데이터프레임 생성
# (copy()를 써서 원본 데이터와 완전히 분리합니다)
df_ecommerce = df[[col for col in target_cols if col in df.columns]].copy()

# 3. CSV로 저장
# encoding='utf-8-sig'는 한글 깨짐 방지용이며, index=False는 불필요한 번호 생성을 막습니다.
df_ecommerce.to_csv('hm_analysis.csv', index=False, encoding='utf-8-sig')

## 저장

In [19]:
# drop=True: 기존의 뒤섞인 인덱스를 삭제
# inplace=True: 변경 사항을 현재 데이터프레임에 바로 적용
# df.reset_index(drop=True, inplace=True)

In [20]:
# index=False: 저장할 때 인덱스 번호를 파일에 포함하지 않음
# df.to_csv('clean_data.csv', index=False)